In [1]:
from data.loader import load_dataset_metatask
from omegaconf import OmegaConf
from methods import get_method_class
import torch 

In [2]:
config = {
    "tasks": ["imagenet1k"], # dataset_name
    "methods": ["knn"],  # Method to use: "asif", "csa", or "cka"
    "csa":{
        "sim_dim": 700,
    },
    "asif":{
        "non_zeros": 800,
    },
    "knn":{
        "num_classes": 1000,
        "k": 30,
        "T": 0.07
    },
    "retrieval":{
        "topk": 5,
        "num_gt": 5,
    },
    "classification":{
    },
    "support_embeddings": None,

    "imagenet1k": {
        "root": "/home/rida.lefdali/work/dataset/imagenet1k/val",
        "loc_val_solution": "/home/rida.lefdali/work/dataset/imagenet1k/LOC_val_solution.csv",
        "loc_synset_mapping": "/home/rida.lefdali/work/dataset/imagenet1k/LOC_synset_mapping.txt",
        "hf_img_embedding_name": "imagenet1k_val_dinov2_dinov2-small_image_embeddings.pkl", 
        "hf_text_embedding_name": "imagenet1k_val_gtr_t5_gtr-t5-xl_text_embeddings.pkl", 
        "hf_repo_id": "ridalefdali/imagenet1k_classification_embeddings", 
        "train_test_ratio": 0.8,
        "seed": 42,
        "split": "large",
        "original_ds_split": "val",
        "generate_embedding": False,
        "metatask": "classification", # only "classification"
    },
  
    "places365": {
        "root": "/home/rida.lefdali/work/dataset/places365_standard/train",
        "filelist_places": "/home/rida.lefdali/work/dataset/places365_standard/places365_train_standard.txt",
        "categories_places": "/home/rida.lefdali/work/dataset/places365_standard/categories_places365.txt",
        "hf_img_embedding_name": "places365_dinov2_dinov2-giant_image_embeddings.pkl", 
        "hf_text_embedding_name": "places365_train_gtr_t5_gtr-t5-large_text_embeddings.pkl", 
        "hf_repo_id": "ridalefdali/places365_classification_embeddings", #"ridalefdali/mscoco_classification_embeddings"
        "train_test_ratio": 0.7,
        "seed": 42,
        "split": "large",
        "original_ds_split": "train",
        "generate_embedding": False,
        "metatask": "classification", # "classification" or  "retrieval"
    },

    "image_embedding_models": ["dinov2"],
    "text_embedding_models": ["all_mpnet_base_v2", "gtr_t5", "sentence_t5"], #"alibaba_gte_en_v1_5", "baai_bge_en_v1_5"],
    "embedding_model": {
        "img_encoder": "dinov2", 
        "text_encoder": "gtr_t5", 
        "image_model_variant": "dinov2-giant",
        "text_model_variant": "gtr-t5-large",
        "batch_size": 128,
    }
}

In [3]:
config = OmegaConf.create(config)

In [4]:
dino_size = ['small', 'base', 'large', 'giant']
results = {}
method_name = "knn"
MethodClass = get_method_class(method_name)
method = MethodClass(**config[method_name])

for size in dino_size:
    OmegaConf.update(
        config, 
        "imagenet1k.hf_img_embedding_name", 
        f"imagenet1k_val_dinov2_dinov2-{size}_image_embeddings.pkl"
        )
    task = load_dataset_metatask("imagenet1k", config)
    predictions = task.run(method, support_embeddings=None)
    targets = task.ground_truth
    targets = torch.Tensor(targets)
    predictions = torch.Tensor(predictions)
    # find the predictions that match the target
    correct = predictions.eq(targets.data.view(-1, 1))
    top1 = correct.narrow(1, 0, 1).sum().item()
    top5 = correct.narrow(1, 0, min(5, 10)).sum().item()  # top5 does not make sense if k < 5
    total = targets.size(0)
    top1 = top1 * 100.0 / total
    top5 = top5 * 100.0 / total

    results[size] = {"top1": top1, "top5": top5}

imagenet1k_val_dinov2_dinov2-small_image(…):   0%|          | 0.00/76.8M [00:00<?, ?B/s]

imagenet1k_val_gtr_t5_gtr-t5-xl_text_emb(…):   0%|          | 0.00/1.54M [00:00<?, ?B/s]

imagenet1k_val_dinov2_dinov2-base_image_(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

imagenet1k_val_dinov2_dinov2-large_image(…):   0%|          | 0.00/205M [00:00<?, ?B/s]

imagenet1k_val_dinov2_dinov2-giant_image(…):   0%|          | 0.00/307M [00:00<?, ?B/s]

In [5]:
import rich

In [6]:
rich.print(results)

{
    'small': {'top1': 59.76, 'top5': 83.06},
    'base': {'top1': 65.59, 'top5': 86.07},
    'large': {'top1': 66.99, 'top5': 87.05},
    'giant': {'top1': 64.3, 'top5': 85.39}
}

In [7]:
mteb_path = '/home/rida.lefdali/work/mteb_benchmark.csv'

In [8]:
import polars as pl

In [9]:
mteb_table = pl.read_csv(mteb_path)

In [51]:
gtr_t5_table = mteb_table.filter(pl.col("Model").str.contains("all-mpnet")).select("Model", "STS","Clustering","Zero-shot","Retrieval", "Classification")

In [52]:
df_with_row_mean = gtr_t5_table.with_columns(
    pl.mean_horizontal(['Retrieval', 'Classification', 'STS']).alias('mean_score')
)

In [53]:
df_with_row_mean

Model,STS,Clustering,Zero-shot,Retrieval,Classification,mean_score
str,f64,f64,str,f64,f64,f64
"""[all-mpnet-base-v2](https://hu…",57.6,40.74,"""100%""",33.8,46.83,46.076667


In [27]:
mteb_table

,Rank (Borda),Model,Zero-shot,Memory Usage (MB),Number of Parameters (B),Embedding Dimensions,Max Tokens,Mean (Task),Mean (TaskType),Bitext Mining,Classification,Clustering,Instruction Reranking,Multilabel Classification,Pair Classification,Reranking,Retrieval,STS
i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1,"""[KaLM-Embedding-Gemma3-12B-251…","""73%""",44884.0,11.76,3840.0,32768.0,72.32,62.51,83.76,77.88,55.77,5.49,33.03,84.73,67.27,75.66,79.02
1,2,"""[llama-embed-nemotron-8b](http…","""99%""",28629.0,7.505,4096.0,32768.0,69.46,61.09,81.72,73.21,54.35,10.82,29.86,83.97,67.78,68.69,79.41
2,3,"""[Qwen3-Embedding-8B](https://h…","""99%""",14433.0,7.567,4096.0,32768.0,70.58,61.69,80.89,74.0,57.65,10.06,28.66,86.4,65.63,70.88,81.08
3,4,"""[gemini-embedding-001](https:/…","""99%""",null,null,3072.0,2048.0,68.37,59.59,79.28,71.82,54.59,5.18,29.16,83.63,65.58,67.71,79.4
4,5,"""[Qwen3-Embedding-4B](https://h…","""99%""",7671.0,4.022,2560.0,32768.0,69.45,60.86,79.36,72.33,57.15,11.56,26.77,85.05,65.08,69.6,80.86
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
375,376,"""[LateOn-Code-pretrain](https:/…","""100%""",568.0,0.149,128.0,8192.0,null,null,null,null,null,null,null,null,null,85.61,null
376,377,"""[LateOn-Code-edge-pretrain](ht…","""100%""",64.0,0.017,48.0,7999.0,null,null,null,null,null,null,null,null,null,74.63,null
377,378,"""[BiCA-base](https://huggingfac…","""100%""",418.0,0.109,768.0,512.0,null,null,null,null,null,null,null,null,null,22.01,null


In [11]:
ms_coco_path = "/home/rida.lefdali/work/dataset/coco2017/annotations/captions_val2017.json"

In [12]:
import polars as pl
import json

In [13]:
with open(ms_coco_path, 'r') as f:
    data = json.load(f)

In [19]:
len(data['images'])

5000